# Semana 12: Práctica. La curva de rendimientos del Tesoro de Estados Unidos

**Curso:** Tópicos de Finanzas Avanzadas (ECON-421, UPAO 2026-20)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JonathanRosasV/topicos-finanzas-upao/blob/main/12_curva_rendimientos/clase12_practica.ipynb)

En la teoría construimos una curva de tres plazos a mano. Aquí hacemos lo mismo con la curva real del Tesoro de Estados Unidos: la observamos en fechas con formas muy distintas, obtenemos las tasas *spot* y *forward* por *bootstrapping*, comprobamos qué tan bien anticipó la pendiente las recesiones, contrastamos la teoría de las expectativas con casi 50 años de datos y medimos lo que un alza de tasas le hizo al precio de los bonos en 2022.

**Uso de este notebook.** Es material de estudio: esta semana no se desarrolla en la sesión (las horas de práctica del 18/11 son las exposiciones del informe país) y no tiene tarea asociada. La muestra está cerrada en diciembre de 2025, de modo que los resultados no cambian con la fecha en que lo ejecutes. La sección 6 aplica a datos históricos las mismas preguntas con las que se leen los indicadores 5, 6 y 7 de la ficha comparativa.

**Requisito previo:** `git pull` en tu fork. Esta semana se agrega a `utils/finanzas.py` la sección "Semana 12": `factores_descuento()`, `precio_con_curva()`, `tasa_par()`, `bootstrap_curva()`, `tasa_forward()` y `duracion()`. Requiere conexión a internet (FRED).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import fred
from utils.finanzas import (factores_descuento, precio_con_curva, tasa_par, bootstrap_curva, tasa_forward, duracion)

FIN = "2025-12-31"                                 # muestra cerrada: los resultados no cambian con la fecha de ejecucion
PLAZOS = {"DGS3MO": 0.25, "DGS1": 1, "DGS2": 2, "DGS3": 3, "DGS5": 5, "DGS7": 7, "DGS10": 10, "DGS20": 20, "DGS30": 30}

def sombrear(ax, rec):
    """Sombrea las recesiones fechadas por el NBER (serie USREC de FRED)."""
    ax.fill_between(rec.index, 0, 1, where=rec.values > 0, transform=ax.get_xaxis_transform(), color="gray", alpha=0.25, lw=0)

## 1. Los datos: la curva en cinco fechas

FRED publica los rendimientos del Tesoro a vencimiento constante para nueve plazos, de 3 meses a 30 años. Son **tasas par** aproximadas: el rendimiento de un bono que cotiza cerca de su valor nominal. Tomamos la última observación de cada mes y miramos cinco fotos elegidas por su forma.

In [ ]:
diaria = pd.DataFrame({plazo: fred.get_series(cod, "2000-01-01", FIN) for cod, plazo in PLAZOS.items()}).dropna()
curva = diaria.resample("ME").last()                                    # ultima observacion de cada mes, en %
print(f"Muestra: {curva.index[0]:%Y-%m} a {curva.index[-1]:%Y-%m} ({len(curva)} meses)\n")

FECHAS = {"2004-06": "empinada: salida de la recesion de 2001, tasa de politica en 1%",
          "2006-12": "plana, levemente invertida: fin del ciclo de alzas de 2004 a 2006",
          "2010-12": "muy empinada: tasa de politica en cero",
          "2023-06": "invertida: alzas de 2022 y 2023",
          "2025-12": "cierre de la muestra"}
fotos = curva.loc[[curva.loc[f].index[-1] for f in FECHAS]]
fotos.index = list(FECHAS)
tabla = fotos[[0.25, 2, 10, 30]].copy()
tabla["pendiente 10a-2a (pb)"] = (fotos[10] - fotos[2]) * 100
print(tabla.round(2).to_string())

fig, ax = plt.subplots(figsize=(9, 4.2))
for f in FECHAS:
    ax.plot(fotos.columns, fotos.loc[f], marker="o", ms=3, label=f)
ax.set_xlabel("plazo (anos)"); ax.set_ylabel("rendimiento (%)"); ax.set_title("Curva del Tesoro de Estados Unidos en cinco fechas")
ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.tight_layout()

**Lectura:** las cinco formas corresponden a cinco momentos del ciclo, tal como dice la lámina "Formas de la curva". Empinada cuando la tasa de política está en su mínimo y se esperan alzas (2004 y 2010); plana o invertida al final de un ciclo de alzas, cuando el tramo corto alcanza o supera al largo (2006 y 2023). Observa qué extremo se mueve más entre fotos: el corto pasa de 0.1 a más de 5 por ciento, mientras el de 30 años se mueve en un rango de dos puntos. La pendiente cambia sobre todo porque se mueve el tramo corto, que sigue a la tasa de política.

## 2. De la curva par a las tasas *spot* y *forward*

Primero una comprobación con los números de la teoría: las tasas par 4.00, 4.49 y 4.97 deben devolver la curva *spot* 4.0, 4.5 y 5.0. Luego aplicamos `bootstrap_curva()` a dos fotos reales, interpolando linealmente las tasas par para tener un plazo por año de 1 a 10. Dos simplificaciones: cupones anuales (los del Tesoro son semestrales) e interpolación lineal (los bancos centrales ajustan curvas suaves, como la de Nelson y Siegel).

In [ ]:
# Comprobacion con el ejemplo de la teoria: tasas par 4.00, 4.49 y 4.97 deben devolver la curva spot 4.0, 4.5 y 5.0
spot_teoria = pd.Series({1: 0.040, 2: 0.045, 3: 0.050})
par_teoria = pd.Series({t: tasa_par(spot_teoria, t) for t in (1, 2, 3)})
print("Ejemplo de la teoria | par:", (par_teoria * 100).round(2).tolist(), "| spot recuperada:", (bootstrap_curva(par_teoria) * 100).round(2).tolist())

def curvas(fila):
    """Par (interpolada a plazos anuales), spot por bootstrapping y forward a un ano, en %."""
    nodos = fila[[1, 2, 3, 5, 7, 10]]
    par = pd.Series(np.interp(range(1, 11), nodos.index.to_numpy(dtype=float), nodos.to_numpy()), index=range(1, 11)) / 100
    spot = bootstrap_curva(par)
    fwd = pd.Series({t: spot[1] if t == 1 else tasa_forward(spot, t - 1, 1) for t in spot.index})
    return pd.DataFrame({"par": par, "spot": spot, "forward a 1 ano": fwd}) * 100

fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for a, f in zip(ax, ("2010-12", "2023-06")):
    c = curvas(fotos.loc[f])
    print(f"\n{f}:"); print(c.round(2).T.to_string())
    c.plot(ax=a, marker="o", ms=3, color=["#000080", "#006E6E", "#B22222"])
    a.set_title(f"Par, spot y forward: {f}"); a.set_xlabel("plazo (anos)"); a.grid(alpha=0.3)
ax[0].set_ylabel("%"); plt.tight_layout()

**Lectura:** con la curva de pendiente positiva (2010), la *spot* queda por encima de la par y la *forward* por encima de ambas; con la curva invertida (2023) el orden se invierte. Es la relación entre un promedio y su valor marginal: la tasa *spot* a $n$ años es un promedio geométrico de las *forward* anuales, de modo que si el promedio sube, la tasa marginal tiene que estar por encima de él. Mira los niveles: en diciembre de 2010, con la tasa a un año en 0.3 por ciento, la curva implicaba tasas a un año superiores a 4 por ciento cinco años después. Las *forward* se ven irregulares en los plazos largos porque amplifican cualquier quiebre de la interpolación; esa es la razón práctica para usar curvas suavizadas.

## 3. La pendiente y las recesiones

La pendiente de 10 años menos 2 años, con promedios mensuales desde 1976. Para cada recesión fechada por el NBER buscamos la primera inversión de la curva en los 36 meses anteriores y contamos los meses de anticipación.

In [ ]:
gs = pd.DataFrame({p: fred.get_series(c, "1976-06-01", FIN) for c, p in {"GS1": 1, "GS2": 2, "GS10": 10}.items()})   # promedios mensuales
rec = fred.get_series("USREC", "1976-06-01", FIN)
pendiente = (gs[10] - gs[2]) * 100                                       # puntos basicos

inv = pendiente < 0
inicios_rec = rec.index[(rec == 1) & (rec.shift(1) == 0)]
filas = []
for r in inicios_rec:
    ventana = inv.loc[r - pd.DateOffset(months=36): r - pd.DateOffset(months=1)]
    if ventana.any():
        f = ventana.index[ventana][0]
        filas.append({"inicio de la recesion": f"{r:%Y-%m}", "primera inversion en los 36 meses previos": f"{f:%Y-%m}",
                      "meses de anticipacion": (r.to_period("M") - f.to_period("M")).n,
                      "minimo de la pendiente (pb)": round(pendiente.loc[f:r].min())})
    else:
        filas.append({"inicio de la recesion": f"{r:%Y-%m}", "primera inversion en los 36 meses previos": "ninguna (promedios mensuales)",
                      "meses de anticipacion": np.nan, "minimo de la pendiente (pb)": round(pendiente.loc[r - pd.DateOffset(months=36): r].min())})
print(pd.DataFrame(filas).to_string(index=False))
ultima = inv.index[inv][-1]
print(f"\nUltimo episodio: pendiente negativa de {inv.loc['2021':].index[inv.loc['2021':]][0]:%Y-%m} a {ultima:%Y-%m}, minimo {pendiente.loc['2021':].min():.0f} pb; sin recesion fechada por el NBER hasta {FIN[:7]}.")
print(f"Meses con pendiente negativa: {inv.mean():.0%} de la muestra | pendiente promedio: {pendiente.mean():.0f} pb")

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(pendiente.index, pendiente, color="#000080", lw=1.2); ax.axhline(0, color="black", lw=0.7); sombrear(ax, rec)
ax.set_ylabel("puntos basicos"); ax.set_title("Pendiente 10 anos menos 2 anos y recesiones (sombreadas)"); ax.grid(alpha=0.3); plt.tight_layout()

**Lectura:** cinco de las seis recesiones de la muestra fueron precedidas por una inversión, con una anticipación de uno a tres años. Ese registro es el origen de la reputación de la curva como indicador adelantado. Ahora los límites, que son los errores 3 y 4 de la lista de la teoría. La anticipación es tan variable (de 17 a 35 meses) que la señal no sirve para fechar nada. La recesión de 2020 no tuvo inversión previa en promedios mensuales: fue una pandemia, no el final de un ciclo monetario. Y la inversión más profunda desde 1981, la de 2022 a 2024, no fue seguida por una recesión dentro de la muestra. La lectura correcta de una curva invertida es la de la teoría: política contractiva y expectativa de recortes. Que los recortes lleguen con o sin recesión es otra pregunta.

## 4. ¿La tasa *forward* predice la tasa futura?

La teoría de las expectativas puras dice que $f(1,1)$ es la tasa a un año que el mercado espera para dentro de un año. Con los rendimientos a 1 y 2 años calculamos esa *forward* cada mes desde 1976 y la comparamos con la tasa a un año que efectivamente se observó doce meses después.

In [ ]:
y1, y2 = gs[1] / 100, gs[2] / 100
fwd_11 = ((1 + y2) ** 2 / (1 + y1) - 1) * 100                             # f(1,1), tratando los rendimientos a 1 y 2 anos como spot
futura = gs[1].shift(-12)                                                # tasa a 1 ano observada doce meses despues
eh = pd.DataFrame({"forward f(1,1)": fwd_11, "tasa a 1 ano un ano despues": futura, "tasa a 1 ano hoy": gs[1]}).dropna()
eh["error"] = eh["forward f(1,1)"] - eh["tasa a 1 ano un ano despues"]

print(f"Muestra: {eh.index[0]:%Y-%m} a {eh.index[-1]:%Y-%m} ({len(eh)} meses)")
print(f"Forward promedio:                      {eh['forward f(1,1)'].mean():.2f}%")
print(f"Tasa a 1 ano observada un ano despues: {eh['tasa a 1 ano un ano despues'].mean():.2f}%")
print(f"Error promedio (forward - observada):  {eh['error'].mean():+.2f} puntos | el forward quedo por encima en {(eh['error'] > 0).mean():.0%} de los meses")
cambio_esp, cambio_obs = eh["forward f(1,1)"] - eh["tasa a 1 ano hoy"], eh["tasa a 1 ano un ano despues"] - eh["tasa a 1 ano hoy"]
b = np.polyfit(cambio_esp, cambio_obs, 1)
print(f"Regresion del cambio observado sobre el cambio implicito: pendiente {b[0]:.2f} (expectativas puras: 1.00) | correlacion {np.corrcoef(cambio_esp, cambio_obs)[0, 1]:.2f}")
print("Error promedio por periodo: " + " | ".join(f"{a} a {z}: {eh.loc[a:z, 'error'].mean():+.2f}" for a, z in (("1976", "1989"), ("1990", "2007"), ("2008", "2020"), ("2021", "2024"))))

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(eh.index, eh["forward f(1,1)"], color="#B22222", lw=1.1, label="forward f(1,1) fijado en t")
ax.plot(eh.index, eh["tasa a 1 ano un ano despues"], color="#000080", lw=1.1, label="tasa a 1 ano observada en t + 12")
ax.set_ylabel("%"); ax.set_title("Lo que el forward anticipaba y lo que ocurrio"); ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.tight_layout()

**Lectura:** en promedio la *forward* quedó cerca de medio punto por encima de la tasa que luego se observó, y por encima en dos de cada tres meses. Esa diferencia sistemática es la **prima por plazo**: quien se compromete a dos años exige una compensación, de modo que la *forward* es expectativa más prima, y no un pronóstico insesgado. La regresión lo confirma: si las expectativas puras se cumplieran, el cambio observado de la tasa se movería uno a uno con el cambio implícito en la curva, y la pendiente estimada es muy inferior a uno. Es el ejemplo numérico de la teoría (una *forward* de 5.0 con prima de 0.4 implica una tasa esperada de 4.6) con datos reales.

El detalle por periodos muestra que la prima no es constante y que el error cambia de signo: entre 2021 y 2024 la *forward* quedó por debajo, porque nadie anticipó la magnitud de las alzas de 2022. De ahí el error común número 2: confundir la tasa *forward* con un pronóstico.

## 5. Permanencia en la curva y riesgo de tasa

Dos cálculos con la curva real. Primero, el **retorno por permanencia**: un cupón cero a dos años comprado en diciembre de 2010, con la curva muy empinada. Segundo, el **riesgo de tasa**: lo que el alza de 2022 le hizo al precio de un bono a 10 años, comparando la aproximación por duración con la revalorización exacta. En el segundo cálculo se revalora el mismo bono al nuevo rendimiento, sin considerar el cupón cobrado ni el año transcurrido, para aislar el efecto de la tasa.

In [ ]:
# (a) Permanencia en la curva: cupon cero a 2 anos comprado a fines de 2010, con la curva sin cambios un ano despues
c10 = curvas(fotos.loc["2010-12"]) / 100
s1, s2 = c10.loc[1, "spot"], c10.loc[2, "spot"]
p0 = 1 / (1 + s2) ** 2
print(f"Diciembre de 2010: spot a 1 ano {s1:.2%}, a 2 anos {s2:.2%}, forward f(1,1) {tasa_forward(c10['spot'], 1, 1):.2%}")
for nombre, tasa in (("curva sin cambios", s1), ("se cumple el forward", tasa_forward(c10["spot"], 1, 1))):
    print(f"  Retorno a un ano del cupon cero a 2 anos si {nombre}: {1 / (1 + tasa) / p0 - 1:.2%} (frente a {s1:.2%} del bono a 1 ano)")

# (b) Riesgo de tasa: el bono a 10 anos en 2022
ini, fin = diaria.loc["2021"].iloc[-1], diaria.loc["2022"].iloc[-1]
y0, yf_ = ini[10] / 100, fin[10] / 100
d = duracion(cupon=y0 * 100, plazo=10, y=y0)                               # bono a la par: cupon igual al rendimiento
aprox = -d["modificada"] * (yf_ - y0)
exacto = duracion(cupon=y0 * 100, plazo=10, y=yf_)["precio"] / d["precio"] - 1   # mismo bono revalorado de inmediato al nuevo rendimiento
print(f"\nRendimiento a 10 anos: {y0:.2%} al cierre de 2021 y {yf_:.2%} al cierre de 2022 (alza de {(yf_ - y0) * 1e4:.0f} pb)")
print(f"Bono a 10 anos a la par: duracion de Macaulay {d['macaulay']:.2f}, modificada {d['modificada']:.2f}")
print(f"Cambio de precio aproximado por duracion: {aprox:.1%} | revalorizacion exacta: {exacto:.1%} | diferencia (convexidad): {exacto - aprox:+.1%}")
for plazo in (2, 5, 30):
    ya, yb = ini[plazo] / 100, fin[plazo] / 100
    dd = duracion(ya * 100, plazo, ya)
    print(f"  Bono a {plazo:>2} anos: duracion modificada {dd['modificada']:5.2f}, alza de {(yb - ya) * 1e4:4.0f} pb, cambio de precio exacto {duracion(ya * 100, plazo, yb)['precio'] / 100 - 1:6.1%}")

**Lectura:** si la curva no cambia, el cupón cero a dos años rinde en un año la tasa *forward* $f(1,1)$, bastante más que el bono a un año: ese exceso es el retorno por permanencia en la curva. Si en cambio la tasa futura resulta ser la *forward*, ambos rinden lo mismo. Comprar el bono largo y venderlo antes del vencimiento es apostar a que las tasas subirán menos de lo que la curva implica, y la sección 4 muestra que, en promedio, esa apuesta pagó la prima por plazo.

El segundo bloque es la otra cara: el riesgo que esa prima remunera. Con una duración modificada cercana a 9, un alza de más de 230 puntos básicos le quitó al bono a 10 años cerca de la quinta parte de su valor, y al de 30 años más de un tercio. La aproximación por duración exagera la pérdida en unos dos puntos: para movimientos grandes la relación entre precio y tasa es convexa, y la duración supone una recta. Fíjate en que el bono a 2 años soportó el alza de tasa más grande y la menor pérdida: lo que importa es el producto de la duración por el cambio de tasa.

## 6. Leer una curva con las preguntas de la ficha

Aplicamos a las cinco fotos las dos primeras preguntas de la guía de los apuntes (sección 8): **nivel**, el rendimiento a 2 años frente a la tasa de referencia, y **pendiente**, con el mismo umbral de 25 puntos básicos de la función `leer_pendiente()` de los apuntes. Como son fechas pasadas, agregamos lo que ocurrió con la tasa de referencia un año después.

In [ ]:
def leer_pendiente(y2, y10, referencia, umbral_pb=25):
    """Nivel del tramo corto frente a la tasa de referencia y signo de la pendiente (umbral de los apuntes)."""
    nivel = "el mercado espera recortes" if y2 < referencia else "el mercado espera alzas"
    pb = (y10 - y2) * 100
    forma = "positiva" if pb > umbral_pb else "negativa" if pb < -umbral_pb else "cercana a cero"
    return nivel, pb, forma

fondos = fred.get_series("FEDFUNDS", "2000-01-01", FIN)
for f in FECHAS:
    ref = fondos.loc[f].iloc[0]
    nivel, pb, forma = leer_pendiente(fotos.loc[f, 2], fotos.loc[f, 10], ref)
    doce = fondos.shift(-12).loc[f].iloc[0] if f < "2025-01" else np.nan
    print(f"{f}: referencia {ref:.2f} | 2 anos {fotos.loc[f, 2]:.2f} | 10 anos {fotos.loc[f, 10]:.2f} | pendiente {pb:+.0f} pb ({forma}) | {nivel}"
          + (f" | referencia un ano despues: {doce:.2f}" if not np.isnan(doce) else ""))

**Lectura:** en junio de 2004 la curva anticipaba alzas y la tasa subió dos puntos en un año; en diciembre de 2006 anticipaba recortes y llegaron. Pero en diciembre de 2010 una curva muy empinada convivió con una tasa de política que siguió en cero cinco años más, y en junio de 2023 el mercado esperaba recortes que tardaron más de un año. La lectura de nivel y pendiente describe lo que el mercado espera y exige en una fecha, con primas incluidas; no lo que va a ocurrir. Por eso la tercera pregunta de la guía es de coherencia: contrastar la lectura de la curva con la fase del ciclo y la postura monetaria identificadas por separado.

## 7. Cierre

Desde esta semana quedan en la librería, en la sección "Semana 12" de `utils/finanzas.py`: `factores_descuento()`, `precio_con_curva()`, `tasa_par()`, `bootstrap_curva()`, `tasa_forward()` y `duracion()`.

**Para practicar por tu cuenta (sin entrega ni nota):**

1. Agrega la fecha "2019-08" al diccionario `FECHAS` y lee esa curva con la sección 6. ¿Qué anticipaba?
2. En la sección 4, repite el contraste con la *forward* $f(2,3)$ usando los rendimientos a 2 y 5 años (series `GS2` y `GS5`) y la tasa a 3 años (`GS3`) observada dos años después.
3. En la sección 5, calcula la pérdida del bono a 10 años ante un alza de 100 puntos básicos partiendo de un rendimiento de 1.5 por ciento y de uno de 5 por ciento. ¿Por qué difieren?

**Próxima semana:** política monetaria y mercados financieros (semana 13).